## Imports and Definitions

In [93]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import xgboost as xgb
from sklearn.metrics import f1_score
import joblib

In [109]:
def compute_rule_coverage(rules_df, len_rows):
    """
    Computes total support, removes duplicate IDs, and sorts rules by percentage.

    :param rules_df: DataFrame containing rule-based results with 'support' and 'ids' columns.
    :param len_rows: Total number of rows in the dataset (for percentage calculations).
    :return: Sorted DataFrame of rules.
    """
    # Compute total support
    total_support = rules_df['support'].sum()
    support_pct = round(total_support / len_rows * 100, 2)
    
    # Remove duplicate IDs
    unique_ids = set(item for sublist in rules_df['ids'].values for item in sublist)
    unique_ids_pct = round(len(unique_ids) / len_rows * 100, 2)
    
    # Print results
    print(f"Total Support: {support_pct}% ({total_support})")
    print("After removing duplicates:")
    print(f"Unique Coverage: {unique_ids_pct}% ({len(unique_ids)})")
    
    # Return sorted DataFrame
    return rules_df.sort_values(by='pct', ascending=False), unique_ids

In [101]:
def get_support(cmc, feature, threshold):
    len_rows = cmc.shape[0]
    results = cmc[cmc[feature] > threshold + 0.0001]['label'].value_counts()
    
    if results.shape[0] != 1:
        #print(f'Incorrect rule for feature: {feature}')
        #print(f'Threshold: {threshold}')
        return None
    
    support = results.values[0]
    pct = round(support / len_rows * 100, 2)
    return support, pct, cmc[cmc[feature] > threshold + 0.0001].index.tolist()

def get_minmax_rules(cmc, analysis_features):
    minmax_rules = pd.DataFrame(columns=['feature', 'is', 'value', 'predict', 'support', 'pct', 'ids'])
    for feature in cmc.columns.drop(analysis_features):
        try:
            # Get the max values per CMC category safely
            max_values = cmc.groupby('CMC')[feature].max().to_dict()

            # Extract values safely, defaulting to -inf (or another placeholder)
            mFN = max_values.get('FN', float('-inf'))
            mFP = max_values.get('FP', float('-inf'))
            mTN = max_values.get('TN', float('-inf'))
            mTP = max_values.get('TP', float('-inf'))
            
            # Compute min/max with safety checks
            min_FT, max_FT = min(mFN, mTN), max(mFP, mTP)
            min_PT, max_PT = min(mFP, mTP), max(mFN, mTN)

        except Exception as e:
            print(f"Error processing feature {feature}: {e}")

        threshold, label = None, None
        
        if min_FT > max_FT:
            threshold, label = min_FT, 0
        elif min_PT > max_PT:
            threshold, label = min_PT, 1
        elif min_PT == max_PT or min_FT == max_FT:
            threshold, label = min_PT, 1

        if threshold is not None:
            result = get_support(cmc, feature, threshold)
            if result:
                support, pct, ids = result
                if support > 2:
                    minmax_rules.loc[len(minmax_rules)] = [feature, '>', threshold, label, support, pct, ids]
    return minmax_rules

In [148]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def get_exclusive_rules(cmc, analysis_features):
    """
    Finds exclusive rules where a feature value uniquely maps to one CMC category.

    :param cmc: DataFrame containing feature values and CMC labels.
    :param analysis_features: List of features to exclude.
    :return: DataFrame containing exclusive rules.
    """
    len_rows = cmc.shape[0]
    exclusivity_rules_list = []

    # Iterate over features while dropping analysis features
    for feature in tqdm(cmc.columns.drop(analysis_features)):
        # Get unique CMC groups for each feature value
        value_to_cmc = cmc.groupby(feature)['CMC'].nunique()

        # Filter values that belong to only one CMC group
        exclusive_values = value_to_cmc[value_to_cmc == 1].index

        # If no exclusive values, skip
        if exclusive_values.empty:
            continue

        # Get mapping of each exclusive value to its corresponding CMC category
        cmc_map = cmc.groupby(feature)['CMC'].first()

        # Get IDs for all exclusive values in one step
        id_map = cmc[cmc[feature].isin(exclusive_values)].groupby(feature).apply(lambda x: x.index.tolist())

        # Construct rules efficiently
        for val in exclusive_values:
            group = cmc_map[val]
            ids = id_map[val]

            if group in ['TP', 'TN']:
                pred = 1 if group == 'TP' else 0
                support = len(ids)
                pct = round(support / len_rows * 100, 2)

                if support > 2:  # Filter out low-support rules
                    exclusivity_rules_list.append([feature, '==', val, pred, support, pct, ids])

    # Convert list to DataFrame at the end for efficiency
    return pd.DataFrame(exclusivity_rules_list, columns=['feature', 'is', 'value', 'predict', 'support', 'pct', 'ids'])

def make_cmc(df, y_train, y_pred_train):
    """Fast CMC computation using NumPy vectorization."""
    train = df.copy()
    
    # Convert target variables to NumPy arrays to avoid broadcasting issues
    y_train = np.array(y_train).flatten()
    y_pred_train = np.array(y_pred_train).flatten()

    # Ensure they have the same length as df
    assert len(y_train) == len(df), "y_train length does not match dataframe rows"
    assert len(y_pred_train) == len(df), "y_pred_train length does not match dataframe rows"

    # Assign labels to the DataFrame
    train['label'] = y_train
    train['predicted'] = y_pred_train

    # Compute CMC categories efficiently using NumPy
    train['CMC'] = np.where(y_pred_train == y_train, 
                            np.where(y_pred_train == 1, 'TP', 'TN'), 
                            np.where(y_pred_train == 1, 'FP', 'FN'))

    return train

In [114]:
def compute_rule_overlap(uids_minmax, uids_ex, len_rows):
    """
    Computes overlap and unique coverage between two rule sets.

    :param uids_minmax: Set of unique IDs from min-max rules.
    :param uids_ex: Set of unique IDs from exclusive rules.
    :param len_rows: Total number of rows in the dataset (for percentage calculations).
    :return: Dictionary with overlap statistics.
    """
    
    # Compute intersection (common IDs in both rule sets)
    intersection_ids = uids_minmax.intersection(uids_ex)
    intersection_count = len(intersection_ids)
    intersection_pct = round(intersection_count / len_rows * 100, 2)
    
    # Compute total unique IDs without double counting overlap
    total_unique_ids = len(uids_minmax) + len(uids_ex) - intersection_count
    total_unique_pct = round(total_unique_ids / len_rows * 100, 2)
    
    # Print results
    print("Overlap")
    print(f"Count: {intersection_count}")
    print(f"Percentage: {intersection_pct}%\n")
    
    print("Without Overlap")
    print(f"Total Unique Count: {total_unique_ids}")
    print(f"Percentage: {total_unique_pct}%")
    
    return {
        "overlap_count": intersection_count,
        "overlap_pct": intersection_pct,
        "unique_count": total_unique_ids,
        "unique_pct": total_unique_pct
    }

## KDD99

In [103]:
# load the data
X_train = pd.read_csv('../Datasets/KDD99/X_train.csv')
X_test = pd.read_csv('../Datasets/KDD99/X_test.csv')
y_train = pd.read_csv('../Datasets/KDD99/y_train.csv')
y_test = pd.read_csv('../Datasets/KDD99/y_test.csv')

In [104]:
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
# evaluate model
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)
print(f'F1 score on training set: {f1_score(y_train, y_pred_train):.2f}')
print(f'F1 score on testing set: {f1_score(y_test, y_pred_test):.2f}')

F1 score on training set: 1.00
F1 score on testing set: 1.00


In [105]:
# save model
full_address = '../Models/KDD99.pkl'
joblib.dump(model, full_address)
print(f'Model saved at {full_address}')

Model saved at ../Models/KDD99.pkl


In [106]:
train_cmc = make_cmc(X_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
train_cmc.to_csv('../Datasets/KDD99/train_cmc.csv', index=False)

In [110]:
minmax_rules, uids_minmax = compute_rule_coverage(get_minmax_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
minmax_rules.to_csv('../Rules/KDD99_minmax_rules.csv', index=False)

Total Support: 0.78% (906)
After removing duplicates:
Unique Coverage: 0.78% (906)


In [111]:
exclusive_rules, uids_ex = compute_rule_coverage(get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
exclusive_rules.to_csv('../Rules/KDD99_exclusive_rules.csv', index=False)

  3%|▎         | 1/39 [00:01<01:08,  1.80s/it]

Processed 0/42 features...


 28%|██▊       | 11/39 [00:43<01:12,  2.59s/it]

Processed 10/42 features...


 54%|█████▍    | 21/39 [00:46<00:08,  2.09it/s]

Processed 20/42 features...


 85%|████████▍ | 33/39 [00:48<00:00,  7.68it/s]

Processed 30/42 features...


100%|██████████| 39/39 [00:48<00:00,  1.25s/it]


Total Support: 119.83% (139559)
After removing duplicates:
Unique Coverage: 70.05% (81589)


In [ ]:
# Example usage
overlap_stats = compute_rule_overlap(uids_minmax, uids_ex, len_rows)

Overlap
Count: 906
Percentage: 0.78%

Without Overlap
Total Unique Count: 81589
Percentage: 70.05%


## UNSW-NB15

In [115]:
import pandas as pd

train = pd.read_csv('../Datasets/UNSW-NB15/processed/le_train.csv')
test = pd.read_csv('../Datasets/UNSW-NB15/processed/le_test.csv')

X_train = train.drop(['id','attack_cat','label'], axis=1)
y_train = train['label'] #0 is normal, 1 is attack

X_test = test.drop(['id','attack_cat','label'], axis=1)
y_test = test['label']  #0 is normal, 1 is attack

In [116]:
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
# evaluate model
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)
print(f'F1 score on training set: {f1_score(y_train, y_pred_train):.2f}')
print(f'F1 score on testing set: {f1_score(y_test, y_pred_test):.2f}')

F1 score on training set: 0.99
F1 score on testing set: 0.85


In [117]:
# save model
import joblib
full_address = '../Models/UNSW-NB15.pkl'
joblib.dump(model, full_address)
print(f'Model saved at {full_address}')

Model saved at ../Models/UNSW-NB15.pkl


In [118]:
train_cmc = make_cmc(X_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
train_cmc.to_csv('../Datasets/UNSW-NB15/train_cmc.csv', index=False)

In [119]:
minmax_rules, uids_minmax = compute_rule_coverage(get_minmax_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
minmax_rules.to_csv('../Rules/UNSWNB15_minmax_rules.csv', index=False)

Total Support: 5.93% (4886)
After removing duplicates:
Unique Coverage: 5.93% (4885)


In [120]:
exclusive_rules, uids_ex = compute_rule_coverage(get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
exclusive_rules.to_csv('../Rules/UNSWNB15_exclusive_rules.csv', index=False)

  5%|▍         | 2/42 [00:45<12:29, 18.74s/it]

Processed 0/45 features...


 21%|██▏       | 9/42 [01:10<04:04,  7.40s/it]

Processed 10/45 features...


 50%|█████     | 21/42 [05:41<07:17, 20.84s/it]

...

 83%|████████▎ | 35/42 [07:09<00:15,  2.17s/it]

Processed 30/45 features...


100%|██████████| 42/42 [07:10<00:00, 10.25s/it]

Processed 40/45 features...


Total Support: 258.65% (212950)
After removing duplicates:
Unique Coverage: 67.67% (55710)


In [121]:
overlap_stats = compute_rule_overlap(uids_minmax, uids_ex, len_rows)

Overlap
Count: 4882
Percentage: 5.93%

Without Overlap
Total Unique Count: 55713
Percentage: 67.67%


## CIC17

In [152]:
import pandas as pd

# load these data
X_train = pd.read_parquet('../Datasets/CIC2017/X_train.parquet')
y_train = pd.read_csv('../Datasets/CIC2017/y_train.csv')
X_test = pd.read_parquet('../Datasets/CIC2017/X_test.parquet')
y_test = pd.read_csv('../Datasets/CIC2017/y_test.csv')

In [153]:
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
# evaluate model
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)
print(f'F1 score on training set: {f1_score(y_train, y_pred_train):.2f}')
print(f'F1 score on testing set: {f1_score(y_test, y_pred_test):.2f}')

F1 score on training set: 1.00
F1 score on testing set: 1.00


In [154]:
# save model
import joblib
full_address = '../Models/CIC17.pkl'
joblib.dump(model, full_address)
print(f'Model saved at {full_address}')

Model saved at ../Models/CIC17.pkl


In [155]:
train_cmc = make_cmc(X_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
train_cmc.to_csv('../Datasets/CIC2017/train_cmc.csv', index=False)

In [156]:
minmax_rules, uids_minmax = compute_rule_coverage(get_minmax_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
minmax_rules.to_csv('../Rules/CIC2017_minmax_rules.csv', index=False)

Total Support: 0.09% (1861)
After removing duplicates:
Unique Coverage: 0.06% (1264)


In [157]:
exclusive_rules, uids_ex = compute_rule_coverage(get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
exclusive_rules.to_csv('../Rules/CIC2017_exclusive_rules.csv', index=False)

  0%|          | 0/78 [00:00<?, ?it/s]

C:\Users\HP\AppData\Local\Temp\ipykernel_26432\1733560491.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  id_map = cmc[cmc[feature].isin(exclusive_values)].groupby(feature).apply(lambda x: x.index.tolist())
  1%|▏         | 1/78 [00:19<24:45, 19.29s/it]C:\Users\HP\AppData\Local\Temp\ipykernel_26432\1733560491.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  id_map = cmc[cmc[feature].isin(exclusive_

Total Support: 1119.42% (22588501)
After removing duplicates:
Unique Coverage: 93.48% (1886245)


In [158]:
# Example usage
overlap_stats = compute_rule_overlap(uids_minmax, uids_ex, len_rows)

Overlap
Count: 1264
Percentage: 0.06%

Without Overlap
Total Unique Count: 1886245
Percentage: 93.48%


## DoH20

In [142]:
import pandas as pd

# load these data
X_train = pd.read_csv('../Datasets/DoH20/X_train.csv')
y_train = pd.read_csv('../Datasets/DoH20/y_train.csv')
X_test = pd.read_csv('../Datasets/DoH20/X_test.csv')
y_test = pd.read_csv('../Datasets/DoH20/y_test.csv')

In [143]:
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
# evaluate model
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)
print(f'F1 score on training set: {f1_score(y_train, y_pred_train):.2f}')
print(f'F1 score on testing set: {f1_score(y_test, y_pred_test):.2f}')

F1 score on training set: 1.00
F1 score on testing set: 1.00


In [144]:
# save model
import joblib
full_address = '../Models/DoH20.pkl'
joblib.dump(model, full_address)
print(f'Model saved at {full_address}')

Model saved at ../Models/DoH20.pkl


In [145]:
train_cmc = make_cmc(X_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
train_cmc.to_csv('../Datasets/DoH20/train_cmc.csv', index=False)

In [146]:
minmax_rules, uids_minmax = compute_rule_coverage(get_minmax_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
minmax_rules.to_csv('../Rules/DOH20_minmax_rules.csv', index=False)

Total Support: 0.0% (0)
After removing duplicates:
Unique Coverage: 0.0% (0)


In [149]:
exclusive_rules, uids_ex = compute_rule_coverage(get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
exclusive_rules.to_csv('../Rules/DOH20_exclusive_rules.csv', index=False)

  0%|          | 0/29 [00:00<?, ?it/s]C:\Users\HP\AppData\Local\Temp\ipykernel_26432\1733560491.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  id_map = cmc[cmc[feature].isin(exclusive_values)].groupby(feature).apply(lambda x: x.index.tolist())
  3%|▎         | 1/29 [00:14<06:52, 14.74s/it]C:\Users\HP\AppData\Local\Temp\ipykernel_26432\1733560491.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  id_

Total Support: 724.18% (1556473)
After removing duplicates:
Unique Coverage: 100.0% (214924)


In [150]:
overlap_stats = compute_rule_overlap(uids_minmax, uids_ex, len_rows)

Overlap
Count: 0
Percentage: 0.0%

Without Overlap
Total Unique Count: 214924
Percentage: 100.0%


## BIG15

In [136]:
# load these data
X_train = pd.read_csv('../Datasets/BIG15/X_train.csv')
X_test = pd.read_csv('../Datasets/BIG15/X_test.csv')
y_train = pd.read_csv('../Datasets/BIG15/y_train.csv')
y_test = pd.read_csv('../Datasets/BIG15/y_test.csv')

In [137]:
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
# evaluate model
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)
print(f'F1 score on training set: {f1_score(y_train, y_pred_train):.2f}')
print(f'F1 score on testing set: {f1_score(y_test, y_pred_test):.2f}')

F1 score on training set: 1.00
F1 score on testing set: 0.99


In [ ]:
# save model
full_address = '../Models/BIG15.pkl'
joblib.dump(model, full_address)
print(f'Model saved at {full_address}')

Model saved at ../Models/BIG15.pkl


In [138]:
train_cmc = make_cmc(X_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
train_cmc.to_csv('../Datasets/BIG15/train_cmc.csv', index=False)

In [139]:
minmax_rules, uids_minmax = compute_rule_coverage(get_minmax_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
minmax_rules.to_csv('../Rules/BIG15_minmax_rules.csv', index=False)

Total Support: 0.12% (133)
After removing duplicates:
Unique Coverage: 0.11% (125)


In [140]:
exclusive_rules, uids_ex = compute_rule_coverage(get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)
exclusive_rules.to_csv('../Rules/BIG15_exclusive_rules.csv', index=False)

  6%|▌         | 3/54 [00:00<00:02, 22.78it/s]

Processed 0/57 features...


 20%|██        | 11/54 [00:16<01:23,  1.95s/it]

Processed 10/57 features...


 39%|███▉      | 21/54 [00:22<00:18,  1.82it/s]

...

Processed 40/57 features...


 94%|█████████▍| 51/54 [04:05<00:22,  7.57s/it]

Processed 50/57 features...


100%|██████████| 54/54 [04:09<00:00,  4.63s/it]


Total Support: 812.73% (897551)
After removing duplicates:
Unique Coverage: 95.35% (105303)


In [141]:
overlap_stats = compute_rule_overlap(uids_minmax, uids_ex, len_rows)

Overlap
Count: 121
Percentage: 0.11%

Without Overlap
Total Unique Count: 105307
Percentage: 95.35%
